In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

base_dir = Path().resolve().parent
sys.path.append(str(base_dir))

from src.data_preprocessing import data_split
from utils.evaluate_model import evaluate_model
from src.shap_plots import compute_shap
from utils.save_results import save_evaluation, save_shap, save_model

RF_MODEL_DIR = str(base_dir / "models_sectors/rf")
RIDGE_MODEL_DIR = str(base_dir / "models_sectors/ridge")
XGB_MODEL_DIR = str(base_dir / "models_sectors/xgb")
RF_MODEL_PARAMS = str(base_dir / "params/rf")
RIDGE_MODEL_PARAMS = str(base_dir / "params/ridge")
XGB_MODEL_PARAMS = str(base_dir / "params/xgb")




In [2]:
FEATURE_COLS = [
    "CPI_Change_lag1", "Rate_Change", "GDP_Growth_lag2",
    "Unemp_Change_lag1", "USD_Change", "VIX_Change",
    "Credit_Spread_lag2",
]

SECTORS = {
    "tech":       "Tech_Return",
    "healthcare": "Healthcare_Return",
    "finance":    "Finance_Return",
    "industrial": "Industrial_Return",
    "energy":     "Energy_Return",
}

# load data
df = pd.read_csv("../data/processed/processed_data.csv", parse_dates=['Date'])
print(f"Data shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.isnull().sum()
df.dropna(inplace=True)  # drop rows with missing values (if any)
print(f"Data shape after dropping NA: {df.shape}")
df.head()

Data shape: (415, 18)
Columns: ['Date', 'SP500_Return', 'Tech_Return', 'Healthcare_Return', 'Finance_Return', 'Industrial_Return', 'Energy_Return', 'CPI_Change', 'Rate_Change', 'GDP_Growth', 'Unemp_Change', 'USD_Change', 'VIX_Change', 'Credit_Spread', 'CPI_Change_lag1', 'GDP_Growth_lag2', 'Unemp_Change_lag1', 'Credit_Spread_lag2']
Data shape after dropping NA: (321, 18)


,Date,SP500_Return,Tech_Return,Healthcare_Return,Finance_Return,Industrial_Return,Energy_Return,CPI_Change,Rate_Change,GDP_Growth,Unemp_Change,USD_Change,VIX_Change,Credit_Spread,CPI_Change_lag1,GDP_Growth_lag2,Unemp_Change_lag1,Credit_Spread_lag2
94,1999-01-01,4.019086,14.756087,4.695698,1.718487,-1.087315,-6.784608,0.3,-0.05,6.149024,-0.1,0.093334,2.556172,1.05,0.3,6.127733,0.0,0.93
95,1999-02-01,-3.281514,-10.444266,0.114613,1.560499,0.896293,-0.863315,0.0,0.13,6.170497,0.1,2.486665,0.787368,0.99,0.3,6.149445,-0.1,1.01
96,1999-03-01,3.806063,7.168039,2.600486,2.924551,1.706203,12.864229,0.1,0.05,6.198760,-0.2,2.335218,-3.513066,0.91,0.0,6.149024,0.1,1.05
97,1999-04-01,3.724186,0.596000,3.509132,6.782267,14.008879,13.804793,1.1,-0.07,6.204344,0.1,1.081926,-1.830994,0.84,0.1,6.170497,-0.2,0.99
98,1999-05-01,-2.528753,0.338983,-3.119268,-6.219939,-1.980263,-2.181911,0.1,0.00,6.173325,-0.1,0.447856,2.725929,0.79,1.1,6.198760,0.1,0.91


In [3]:
import optuna
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error

optuna.logging.set_verbosity(optuna.logging.WARNING)

## Train random forest models for sectors ###

In [4]:
from sklearn.metrics import mean_squared_error, r2_score

from utils.save_results import save_feature_importance

N_SEEDS = 5          # for RF stability analysis
N_CV_SPLITS = 3      # walk-forward folds inside training data
N_TRIALS = 100

all_rf_params  = {}
all_rf_results = {}
all_rf_studies = {}  # keep for appendix / convergence plots

for sector_name, return_col in SECTORS.items():

    print(f"\n{'='*55}\n  {sector_name.upper()} → {return_col}\n{'='*55}")

    splits = data_split(df, FEATURE_COLS, return_col)
    X_train, y_train = splits["X_train"], splits["y_train"]
    X_test,  y_test  = splits["X_test"],  splits["y_test"]
    print(f"  Train: {len(X_train)} | Test: {len(X_test)}")


    # Optuna tuning on walk-forward CV inside TRAIN only
    tscv = TimeSeriesSplit(n_splits=N_CV_SPLITS)

    def objective(trial):
        params = {
            "n_estimators":      trial.suggest_int("n_estimators", 30, 100, step=5),
            "max_depth":         trial.suggest_int("max_depth", 2, 6),
            "min_samples_split": trial.suggest_int("min_samples_split", 5, 50),
            "min_samples_leaf":  trial.suggest_int("min_samples_leaf", 3, 30),
            "max_features":      trial.suggest_categorical(
                "max_features", ["sqrt", "log2", 0.3, 0.5, 0.7]),
            "max_samples":       trial.suggest_float("max_samples", 0.3, 0.9),
            "n_jobs":       -1,
            "random_state":  42,
        }

        fold_rmses = []
        for tr_idx, val_idx in tscv.split(X_train):
            X_tr, X_val = X_train[tr_idx], X_train[val_idx]
            y_tr, y_val = y_train[tr_idx], y_train[val_idx]
            m = RandomForestRegressor(**params)
            m.fit(X_tr, y_tr)
            fold_rmses.append(
                np.sqrt(mean_squared_error(y_val, m.predict(X_val))))
        return float(np.mean(fold_rmses))  # mean CV RMSE, NOT test RMSE

    study = optuna.create_study(
        direction="minimize",
        study_name=f"rf_{sector_name}",
        sampler=optuna.samplers.TPESampler(seed=42),
    )
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
    all_rf_studies[sector_name] = study

    best = {**study.best_params, "n_jobs": -1}
    print(f"  Best CV RMSE: {study.best_value:.4f}")
    print(f"  Params: {best}")

    # Refit with N seeds on full training data
    seed_preds_test  = []
    seed_preds_train = []
    for seed in range(N_SEEDS):
        m = RandomForestRegressor(**{**best, "random_state": seed})
        m.fit(X_train, y_train)
        seed_preds_test.append(m.predict(X_test))
        seed_preds_train.append(m.predict(X_train))

    test_pred_mean  = np.mean(seed_preds_test,  axis=0)
    train_pred_mean = np.mean(seed_preds_train, axis=0)

    seed_rmses = [np.sqrt(mean_squared_error(y_test, p)) for p in seed_preds_test]
    seed_r2s   = [r2_score(y_test, p)                    for p in seed_preds_test]
    print(f"  RF across {N_SEEDS} seeds → "
          f"RMSE {np.mean(seed_rmses):.4f} ± {np.std(seed_rmses):.4f} | "
          f"R² {np.mean(seed_r2s):+.4f} ± {np.std(seed_r2s):.4f}")

    # Final model = the median-seed one (reproducible pick for SHAP + saving)
    median_idx = int(np.argsort(seed_rmses)[len(seed_rmses) // 2])
    final_model = RandomForestRegressor(
        **{**best, "random_state": median_idx}).fit(X_train, y_train)

    all_rf_params[sector_name] = {**best, "random_state": median_idx}

    # Evaluate, SHAP, save
    ev_result = evaluate_model(
        final_model, X_train, y_train, X_test, y_test, f"RF - {sector_name}")

    shap_test_df, mean_abs_shap = compute_shap(
        final_model, X_train, X_test, FEATURE_COLS, model_type="tree")

    save_evaluation(sector_name, ev_result, FEATURE_COLS, y_test, RF_MODEL_DIR)
    save_shap(sector_name, shap_test_df, mean_abs_shap, FEATURE_COLS, RF_MODEL_DIR)
    save_model(sector_name, final_model, RF_MODEL_DIR)
    save_feature_importance(sector_name, final_model, FEATURE_COLS, RF_MODEL_DIR)

    all_rf_results[sector_name] = {
        "cv_rmse":        study.best_value,
        "test_r2":        ev_result[1]["r2"],
        "test_rmse":      ev_result[1]["rmse"],
        "test_dir":       ev_result[1]["dir"],
        # "seed_rmse_mean": float(np.mean(seed_rmses)),
        # "seed_rmse_std":  float(np.std(seed_rmses)),
        # "seed_r2_mean":   float(np.mean(seed_r2s)),
        # "seed_r2_std":    float(np.std(seed_r2s)),
    }

print("\n ALL SECTORS TRAINED AND SAVED")


  TECH → Tech_Return

  Features (7): CPI_Change_lag1, Rate_Change, GDP_Growth_lag2, Unemp_Change_lag1, USD_Change, VIX_Change, Credit_Spread_lag2
  Train: 252 | Test: 69


Best trial: 88. Best value: 4.01609: 100%|██████████| 100/100 [00:16<00:00,  6.10it/s]


  Best CV RMSE: 4.0161
  Params: {'n_estimators': 65, 'max_depth': 6, 'min_samples_split': 29, 'min_samples_leaf': 3, 'max_features': 'log2', 'max_samples': 0.7956420410386159, 'n_jobs': -1}
  RF across 5 seeds → RMSE 5.6788 ± 0.0277 | R² +0.1831 ± 0.0080
  RF - tech
  Metric                         Train       Test
  ---------------------------------------------
  R²                            0.3801     0.1824
  RMSE (%)                       5.291      5.682
  MAE (%)                        3.740      4.642
  Directional Acc (%)            74.21      68.12
  Base value (mean prediction) : 0.4907
Saved evaluation  → /Users/macbook/Desktop/macroeconomic-impact-stock-ml/models_sectors/rf/results/tech_results.pkl
R² train=0.3801  test=0.1824
  Saved SHAP        → /Users/macbook/Desktop/macroeconomic-impact-stock-ml/models_sectors/rf/shap/tech_shap.pkl
    Features: 7
  Saved model       → /Users/macbook/Desktop/macroeconomic-impact-stock-ml/models_sectors/rf/trained/tech_model.pkl  (92.

Best trial: 32. Best value: 3.37854: 100%|██████████| 100/100 [00:15<00:00,  6.48it/s]


  Best CV RMSE: 3.3785
  Params: {'n_estimators': 60, 'max_depth': 6, 'min_samples_split': 10, 'min_samples_leaf': 3, 'max_features': 0.3, 'max_samples': 0.6730762608603014, 'n_jobs': -1}
  RF across 5 seeds → RMSE 4.1321 ± 0.0322 | R² +0.0741 ± 0.0145
  RF - healthcare
  Metric                         Train       Test
  ---------------------------------------------
  R²                            0.4734     0.0724
  RMSE (%)                       2.915      4.136
  MAE (%)                        2.259      3.272
  Directional Acc (%)            75.40      60.87
  Base value (mean prediction) : 0.6082
Saved evaluation  → /Users/macbook/Desktop/macroeconomic-impact-stock-ml/models_sectors/rf/results/healthcare_results.pkl
R² train=0.4734  test=0.0724
  Saved SHAP        → /Users/macbook/Desktop/macroeconomic-impact-stock-ml/models_sectors/rf/shap/healthcare_shap.pkl
    Features: 7
  Saved model       → /Users/macbook/Desktop/macroeconomic-impact-stock-ml/models_sectors/rf/trained/healt

...

  Best CV RMSE: 5.5331
  Params: {'n_estimators': 60, 'max_depth': 6, 'min_samples_split': 11, 'min_samples_leaf': 3, 'max_features': 0.7, 'max_samples': 0.5402926116811306, 'n_jobs': -1}
  RF across 5 seeds → RMSE 5.7703 ± 0.0597 | R² +0.1906 ± 0.0167
  RF - finance
  Metric                         Train       Test
  ---------------------------------------------
  R²                            0.4392     0.1914
  RMSE (%)                       4.621      5.768
  MAE (%)                        3.228      4.392
  Directional Acc (%)            73.81      59.42
  Base value (mean prediction) : 0.2365
Saved evaluation  → /Users/macbook/Desktop/macroeconomic-impact-stock-ml/models_sectors/rf/results/finance_results.pkl
R² train=0.4392  test=0.1914
  Saved SHAP        → /Users/macbook/Desktop/macroeconomic-impact-stock-ml/models_sectors/rf/shap/finance_shap.pkl
    Features: 7
  Saved model       → /Users/macbook/Desktop/macroeconomic-impact-stock-ml/models_sectors/rf/trained/finance_model.

Best trial: 73. Best value: 4.30408: 100%|██████████| 100/100 [00:18<00:00,  5.45it/s]


  Best CV RMSE: 4.3041
  Params: {'n_estimators': 95, 'max_depth': 4, 'min_samples_split': 23, 'min_samples_leaf': 3, 'max_features': 0.7, 'max_samples': 0.8422920902190252, 'n_jobs': -1}
  RF across 5 seeds → RMSE 5.1435 ± 0.0177 | R² +0.2769 ± 0.0050
  RF - industrial
  Metric                         Train       Test
  ---------------------------------------------
  R²                            0.4362     0.2799
  RMSE (%)                       3.952      5.133
  MAE (%)                        2.901      4.090
  Directional Acc (%)            75.79      62.32
  Base value (mean prediction) : 0.4455
Saved evaluation  → /Users/macbook/Desktop/macroeconomic-impact-stock-ml/models_sectors/rf/results/industrial_results.pkl
R² train=0.4362  test=0.2799
  Saved SHAP        → /Users/macbook/Desktop/macroeconomic-impact-stock-ml/models_sectors/rf/shap/industrial_shap.pkl
    Features: 7
  Saved model       → /Users/macbook/Desktop/macroeconomic-impact-stock-ml/models_sectors/rf/trained/indus

Best trial: 73. Best value: 5.38861: 100%|██████████| 100/100 [00:15<00:00,  6.36it/s]


  Best CV RMSE: 5.3886
  Params: {'n_estimators': 55, 'max_depth': 4, 'min_samples_split': 7, 'min_samples_leaf': 3, 'max_features': 0.7, 'max_samples': 0.7946948126172158, 'n_jobs': -1}
  RF across 5 seeds → RMSE 9.7704 ± 0.0939 | R² +0.1135 ± 0.0170
  RF - energy
  Metric                         Train       Test
  ---------------------------------------------
  R²                            0.4765     0.1069
  RMSE (%)                       4.477      9.807
  MAE (%)                        3.569      6.929
  Directional Acc (%)            70.63      49.28
  Base value (mean prediction) : 0.4029
Saved evaluation  → /Users/macbook/Desktop/macroeconomic-impact-stock-ml/models_sectors/rf/results/energy_results.pkl
R² train=0.4765  test=0.1069
  Saved SHAP        → /Users/macbook/Desktop/macroeconomic-impact-stock-ml/models_sectors/rf/shap/energy_shap.pkl
    Features: 7
  Saved model       → /Users/macbook/Desktop/macroeconomic-impact-stock-ml/models_sectors/rf/trained/energy_model.pkl  

## Train ridge models for sectors ###

In [5]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
import numpy as np
import pandas as pd


N_SPLITS = 3
tscv = TimeSeriesSplit(n_splits=N_SPLITS)

all_ridge_results = {}

for sector_name, return_col in SECTORS.items():

    print(f"\n{'='*55}\n  RIDGE — {sector_name.upper()} → {return_col}\n{'='*55}")

    splits  = data_split(df, FEATURE_COLS, return_col)
    X_train, y_train = splits["X_train"], splits["y_train"]
    X_test,  y_test  = splits["X_test"],  splits["y_test"]
    print(f"  Train: {len(X_train)} | Test: {len(X_test)}")

    # Tune alpha on walk-forward CV
    alphas = np.logspace(-3, 5, 100)
    cv_rmses = []
    for alpha in alphas:
        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("ridge",  Ridge(alpha=alpha)),
        ])
        # neg_root_mean_squared_error → flip sign so lower = better
        scores = cross_val_score(
            pipe, X_train, y_train,
            cv=tscv,
            scoring="neg_root_mean_squared_error",
        )
        cv_rmses.append(-scores.mean())
    cv_rmses = np.array(cv_rmses)

    best_idx   = np.argmin(cv_rmses)
    best_alpha = alphas[best_idx]
    best_cv    = cv_rmses[best_idx]
    print(f"  Best alpha:    {best_alpha:.4f}")
    print(f"  Best CV RMSE:  {best_cv:.4f}")

    # Final model: refit on full training data
    final_pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge",  Ridge(alpha=best_alpha)),
    ])
    final_pipe.fit(X_train, y_train)

    # Evaluate (evaluate_model should accept the pipeline directly —
    # pipe.predict(X) handles scaling internally)
    ev_result = evaluate_model(
        final_pipe, X_train, y_train, X_test, y_test,
        f"Ridge - {sector_name}")

    # Coefficients
    ridge_step = final_pipe.named_steps["ridge"]
    coef_df = pd.DataFrame({
        "Feature":     FEATURE_COLS,
        "Coefficient": ridge_step.coef_,
        "Abs_Coef":    np.abs(ridge_step.coef_),
    }).sort_values("Abs_Coef", ascending=False)
    print(f"\n  Standardised coefficients (comparable across features):")
    print(coef_df.to_string(index=False))

    # SHAP
    # Pass the SCALED data to the SHAP linear explainer,
    # otherwise SHAP attributes importance to the unscaled feature space
    scaler  = final_pipe.named_steps["scaler"]
    X_train_scaled = scaler.transform(X_train)
    X_test_scaled  = scaler.transform(X_test)

    shap_test_df, mean_abs_shap = compute_shap(
        ridge_step, X_train_scaled, X_test_scaled, FEATURE_COLS,
        model_type="linear")

    # Save
    save_evaluation(sector_name, ev_result, FEATURE_COLS, y_test, RIDGE_MODEL_DIR)
    save_shap(sector_name, shap_test_df, mean_abs_shap, FEATURE_COLS, RIDGE_MODEL_DIR)
    save_model(sector_name, final_pipe, RIDGE_MODEL_DIR)  
    save_feature_importance(sector_name, final_model, FEATURE_COLS, RIDGE_MODEL_DIR)

    all_ridge_results[sector_name] = {
        "best_alpha": best_alpha,
        "cv_rmse":    best_cv,                
        "test_r2":    ev_result[1]["r2"],
        "test_rmse":  ev_result[1]["rmse"],
        "test_dir":   ev_result[1]["dir"],
    }

print("\n ALL SECTORS TRAINED (RIDGE)")


  RIDGE — TECH → Tech_Return

  Features (7): CPI_Change_lag1, Rate_Change, GDP_Growth_lag2, Unemp_Change_lag1, USD_Change, VIX_Change, Credit_Spread_lag2
  Train: 252 | Test: 69
  Best alpha:    102.3531
  Best CV RMSE:  4.0663
  Ridge - tech
  Metric                         Train       Test
  ---------------------------------------------
  R²                            0.2429    -0.1874
  RMSE (%)                       5.847      6.847
  MAE (%)                        4.166      4.898
  Directional Acc (%)            70.63      69.57

  Standardised coefficients (comparable across features):
           Feature  Coefficient  Abs_Coef
        VIX_Change    -2.400620  2.400620
 Unemp_Change_lag1    -0.483878  0.483878
        USD_Change    -0.306759  0.306759
   CPI_Change_lag1     0.179611  0.179611
   GDP_Growth_lag2    -0.140593  0.140593
Credit_Spread_lag2    -0.101077  0.101077
       Rate_Change     0.083933  0.083933
  Base value (mean prediction) : 0.4196
Saved evaluation  → /U

In [6]:
summary = pd.DataFrame(all_rf_results).T
summary.columns = ["best_rmse", "test_r2", "test_rmse",'test_dir']
summary = summary.sort_values("test_r2", ascending=False)
print(summary.round(4).to_string())

            best_rmse  test_r2  test_rmse  test_dir
industrial     4.3041   0.2799     5.1331   62.3188
finance        5.5331   0.1914     5.7676   59.4203
tech           4.0161   0.1824     5.6815   68.1159
energy         5.3886   0.1069     9.8070   49.2754
healthcare     3.3785   0.0724     4.1361   60.8696


In [7]:
summary = pd.DataFrame(all_ridge_results).T
summary.columns = ["best_alpha", "cv_r2", "test_r2", "test_rmse", 'test_dir']

summary = summary.sort_values("test_r2", ascending=False)
print(summary.round(4).to_string())

            best_alpha   cv_r2  test_r2  test_rmse  test_dir
energy         23.1013  5.4274   0.2217     9.1551   56.5217
finance        10.9750  5.4165   0.0720     6.1790   59.4203
industrial     27.8256  4.2454  -0.0541     6.2101   63.7681
tech          102.3531  4.0663  -0.1874     6.8467   69.5652
healthcare     40.3702  3.2934  -0.3561     5.0009   56.5217


In [8]:
params_df = pd.DataFrame(all_rf_params).T
print("\nOptuna chose these hyperparameters per sector:")
print(params_df.to_string())

# save for reference
import pickle
import os

os.makedirs(RF_MODEL_PARAMS, exist_ok=True)
with open(f"{RF_MODEL_PARAMS}/all_params.pkl", "wb") as f:
    pickle.dump(all_rf_params, f)


Optuna chose these hyperparameters per sector:
           n_estimators max_depth min_samples_split min_samples_leaf max_features max_samples n_jobs random_state
tech                 65         6                29                3         log2    0.795642     -1            4
healthcare         60.0       6.0              10.0              3.0          0.3    0.673076   -1.0          3.0
finance            60.0       6.0              11.0              3.0          0.7    0.540293   -1.0          0.0
industrial         95.0       4.0              23.0              3.0          0.7    0.842292   -1.0          2.0
energy             55.0       4.0               7.0              3.0          0.7    0.794695   -1.0          0.0


## Train XGBoost models for sectors ###

In [9]:

import xgboost as xgb


# Same splitter as Ridge and RF
N_SPLITS = 3
tscv = TimeSeriesSplit(n_splits=N_SPLITS)
N_TRIALS = 100

all_xgb_params  = {}
all_xgb_results = {}
all_xgb_studies = {}

for sector_name, return_col in SECTORS.items():

    print(f"\n{'='*55}\n  XGB — {sector_name.upper()} → {return_col}\n{'='*55}")

    splits = data_split(df, FEATURE_COLS, return_col)
    X_train, y_train = splits["X_train"], splits["y_train"]
    X_test,  y_test  = splits["X_test"],  splits["y_test"]
    print(f"  Train: {len(X_train)} | Test: {len(X_test)}")

    dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=FEATURE_COLS)
    dtest  = xgb.DMatrix(X_test,  label=y_test,  feature_names=FEATURE_COLS)

    # Walk-forward fold indices for this sector's training data
    cv_folds = list(tscv.split(np.arange(dtrain.num_row())))

    # Optuna tuning on walk-forward CV
    def objective(trial):
        params = {
            "objective":        "reg:squarederror",
            "eval_metric":      "rmse",
            "verbosity":        0,
            "seed":             42,
            "eta":              trial.suggest_float("eta", 0.005, 0.3, log=True),
            "max_depth":        trial.suggest_int("max_depth", 2, 5),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 25),
            "subsample":        trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "gamma":            trial.suggest_float("gamma", 0.0, 5.0),
        }

        cv_results = xgb.cv(
            params=params,
            dtrain=dtrain,
            num_boost_round=2000,
            folds=cv_folds,
            metrics="rmse",
            early_stopping_rounds=30,
            verbose_eval=False,
        )
        return cv_results["test-rmse-mean"].min()

    study = optuna.create_study(
        direction="minimize",
        study_name=f"xgb_{sector_name}",
        sampler=optuna.samplers.TPESampler(seed=42),
    )
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
    all_xgb_studies[sector_name] = study

    best_params = {
        **study.best_params,
        "objective":   "reg:squarederror",
        "eval_metric": "rmse",
        "verbosity":   0,
        "seed":        42,
    }
    print(f"  Best CV RMSE: {study.best_value:.4f}")
    print(f"  Params: {best_params}")

    # Determine best num_boost_round via CV (no peeking at test)
    cv_results = xgb.cv(
        params=best_params,
        dtrain=dtrain,
        num_boost_round=2000,
        folds=cv_folds,
        metrics="rmse",
        early_stopping_rounds=30,
        verbose_eval=False,
    )
    best_round = int(cv_results["test-rmse-mean"].idxmin()) + 1
    print(f"  Best num_boost_round (from CV): {best_round}")

    # Final model: train on full training data, evaluate once on test
    final_model = xgb.train(
        params=best_params,
        dtrain=dtrain,
        num_boost_round=best_round,
        evals=[(dtrain, "train"), (dtest, "test")],
        verbose_eval=False,
    )

    ev_result = evaluate_model(
        final_model, dtrain, y_train, dtest, y_test, f"XGB - {sector_name}")

    # SHAP
    shap_test_df, mean_abs_shap = compute_shap(
        final_model, X_train, X_test, FEATURE_COLS, model_type="tree")

    # Save
    all_xgb_params[sector_name] = {**best_params, "num_boost_round": best_round}

    save_evaluation(sector_name, ev_result, FEATURE_COLS, y_test, XGB_MODEL_DIR)
    save_shap(sector_name, shap_test_df, mean_abs_shap, FEATURE_COLS, XGB_MODEL_DIR)
    save_model(sector_name, final_model, XGB_MODEL_DIR)
    save_feature_importance(sector_name, final_model, FEATURE_COLS, XGB_MODEL_DIR)

    all_xgb_results[sector_name] = {
        "cv_rmse":   study.best_value,
        "test_r2":   ev_result[1]["r2"],
        "test_rmse": ev_result[1]["rmse"],
        "test_dir":  ev_result[1]["dir"],
        "best_round": best_round,
    }

print("\n ALL SECTORS TRAINED AND SAVED (XGB)")


  XGB — TECH → Tech_Return

  Features (7): CPI_Change_lag1, Rate_Change, GDP_Growth_lag2, Unemp_Change_lag1, USD_Change, VIX_Change, Credit_Spread_lag2
  Train: 252 | Test: 69


Best trial: 98. Best value: 3.96436: 100%|██████████| 100/100 [00:47<00:00,  2.10it/s]


  Best CV RMSE: 3.9644
  Params: {'eta': 0.008714347804365644, 'max_depth': 2, 'min_child_weight': 3, 'subsample': 0.5921988854607231, 'colsample_bytree': 0.924927555782599, 'gamma': 3.3405125759077894, 'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'verbosity': 0, 'seed': 42}
  Best num_boost_round (from CV): 176
  XGB - tech
  Metric                         Train       Test
  ---------------------------------------------
  R²                            0.3450     0.2371
  RMSE (%)                       5.438      5.488
  MAE (%)                        3.877      4.451
  Directional Acc (%)            73.41      71.01
  Base value (mean prediction) : 0.4674
Saved evaluation  → /Users/macbook/Desktop/macroeconomic-impact-stock-ml/models_sectors/xgb/results/tech_results.pkl
R² train=0.3450  test=0.2371
  Saved SHAP        → /Users/macbook/Desktop/macroeconomic-impact-stock-ml/models_sectors/xgb/shap/tech_shap.pkl
    Features: 7
  Saved model       → /Users/macbook/Desktop/macr

Best trial: 92. Best value: 3.28721: 100%|██████████| 100/100 [01:07<00:00,  1.47it/s]


  Best CV RMSE: 3.2872
  Params: {'eta': 0.006585132940304291, 'max_depth': 2, 'min_child_weight': 1, 'subsample': 0.5158630964164258, 'colsample_bytree': 0.5824047189351675, 'gamma': 3.3827410997487264, 'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'verbosity': 0, 'seed': 42}
  Best num_boost_round (from CV): 607
  XGB - healthcare
  Metric                         Train       Test
  ---------------------------------------------
  R²                            0.4950     0.0862
  RMSE (%)                       2.855      4.105
  MAE (%)                        2.258      3.243
  Directional Acc (%)            77.38      60.87
  Base value (mean prediction) : 0.5525
Saved evaluation  → /Users/macbook/Desktop/macroeconomic-impact-stock-ml/models_sectors/xgb/results/healthcare_results.pkl
R² train=0.4950  test=0.0862
  Saved SHAP        → /Users/macbook/Desktop/macroeconomic-impact-stock-ml/models_sectors/xgb/shap/healthcare_shap.pkl
    Features: 7
  Saved model       → /Users/m

...

  Best CV RMSE: 5.4956
  Params: {'eta': 0.03813735241307445, 'max_depth': 4, 'min_child_weight': 12, 'subsample': 0.5561414726262051, 'colsample_bytree': 0.8149945419110202, 'gamma': 1.4421077321927105, 'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'verbosity': 0, 'seed': 42}
  Best num_boost_round (from CV): 69
  XGB - finance
  Metric                         Train       Test
  ---------------------------------------------
  R²                            0.4274     0.1753
  RMSE (%)                       4.669      5.825
  MAE (%)                        3.199      4.271
  Directional Acc (%)            73.81      62.32
  Base value (mean prediction) : 0.2221
Saved evaluation  → /Users/macbook/Desktop/macroeconomic-impact-stock-ml/models_sectors/xgb/results/finance_results.pkl
R² train=0.4274  test=0.1753
  Saved SHAP        → /Users/macbook/Desktop/macroeconomic-impact-stock-ml/models_sectors/xgb/shap/finance_shap.pkl
    Features: 7
  Saved model       → /Users/macbook/Des

Best trial: 98. Best value: 4.22377: 100%|██████████| 100/100 [01:00<00:00,  1.65it/s]


  Best CV RMSE: 4.2238
  Params: {'eta': 0.02480202737061276, 'max_depth': 2, 'min_child_weight': 2, 'subsample': 0.5911958951003784, 'colsample_bytree': 0.9996794558053655, 'gamma': 4.296096762420689, 'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'verbosity': 0, 'seed': 42}
  Best num_boost_round (from CV): 141
  XGB - industrial
  Metric                         Train       Test
  ---------------------------------------------
  R²                            0.5220     0.3182
  RMSE (%)                       3.639      4.995
  MAE (%)                        2.710      3.949
  Directional Acc (%)            76.59      62.32
  Base value (mean prediction) : 0.5491
Saved evaluation  → /Users/macbook/Desktop/macroeconomic-impact-stock-ml/models_sectors/xgb/results/industrial_results.pkl
R² train=0.5220  test=0.3182
  Saved SHAP        → /Users/macbook/Desktop/macroeconomic-impact-stock-ml/models_sectors/xgb/shap/industrial_shap.pkl
    Features: 7
  Saved model       → /Users/mac

Best trial: 52. Best value: 5.31378: 100%|██████████| 100/100 [00:34<00:00,  2.89it/s]


  Best CV RMSE: 5.3138
  Params: {'eta': 0.05460871273526048, 'max_depth': 3, 'min_child_weight': 8, 'subsample': 0.7637391646872494, 'colsample_bytree': 0.9140064272887378, 'gamma': 3.724692118625917, 'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'verbosity': 0, 'seed': 42}
  Best num_boost_round (from CV): 65
  XGB - energy
  Metric                         Train       Test
  ---------------------------------------------
  R²                            0.5842     0.0950
  RMSE (%)                       3.990      9.873
  MAE (%)                        3.192      7.104
  Directional Acc (%)            75.00      50.72
  Base value (mean prediction) : 0.4296
Saved evaluation  → /Users/macbook/Desktop/macroeconomic-impact-stock-ml/models_sectors/xgb/results/energy_results.pkl
R² train=0.5842  test=0.0950
  Saved SHAP        → /Users/macbook/Desktop/macroeconomic-impact-stock-ml/models_sectors/xgb/shap/energy_shap.pkl
    Features: 7
  Saved model       → /Users/macbook/Desktop/